In [ ]:
import os

import pandas as pd
import numpy as np
import re

import random

from datetime import date
from dateutil.relativedelta import relativedelta

data_path = os.path.join("..", "data")
processed_data_path = os.path.join(data_path, 'processed_data/')

## Import Data

### Inclusion Patients

In [ ]:
inclusion_patients_df = pd.read_csv(os.path.join(data_path, "opt_in_ckd_inclusion_patients.csv"))

inclusion_patients_df['imd_decile'] = inclusion_patients_df['imd_decile'].apply(lambda x: pd.NA if pd.isnull(x) else int(x))
inclusion_patients_df['imd_rank'] = inclusion_patients_df['imd_rank'].apply(lambda x: pd.NA if pd.isnull(x) else int(x))

inclusion_patients_df['dateOfBirth'] = pd.to_datetime(inclusion_patients_df['dateOfBirth']).dt.date
inclusion_patients_df['dateOfDeath'] = pd.to_datetime(inclusion_patients_df['dateOfDeath']).dt.date
inclusion_patients_df['inclusion_date'] = pd.to_datetime(inclusion_patients_df['inclusion_date']).dt.date
inclusion_patients_df['endpoint_date'] = pd.to_datetime(inclusion_patients_df['endpoint_date']).dt.date

### Date Grouping Breakdown

In [ ]:
dates_breakdown_df = pd.read_csv(os.path.join(data_path, "ckd_patients_datebreakdown.csv"))

dates_breakdown_df['inclusion_date'] = pd.to_datetime(dates_breakdown_df['inclusion_date']).dt.date
dates_breakdown_df['endpoint_date'] = pd.to_datetime(dates_breakdown_df['endpoint_date']).dt.date
dates_breakdown_df['start_date'] = pd.to_datetime(dates_breakdown_df['start_date']).dt.date
dates_breakdown_df['end_date'] = pd.to_datetime(dates_breakdown_df['end_date']).dt.date

### Lab Results

In [ ]:
lab_results_df = pd.read_csv(os.path.join(processed_data_path, "20260306_processed_lab_results.csv"))

lab_results_df['inclusion_date'] = pd.to_datetime(lab_results_df['inclusion_date']).dt.date
lab_results_df['endpoint_date'] = pd.to_datetime(lab_results_df['endpoint_date']).dt.date
lab_results_df['start_date'] = pd.to_datetime(lab_results_df['start_date']).dt.date
lab_results_df['end_date'] = pd.to_datetime(lab_results_df['end_date']).dt.date

### Activity Data

In [ ]:
activity_data_df = pd.read_csv(os.path.join(processed_data_path, "20260306_processed_activity_data.csv"))

activity_data_df['inclusion_date'] = pd.to_datetime(activity_data_df['inclusion_date']).dt.date
activity_data_df['endpoint_date'] = pd.to_datetime(activity_data_df['endpoint_date']).dt.date
activity_data_df['start_date'] = pd.to_datetime(activity_data_df['start_date']).dt.date
activity_data_df['end_date'] = pd.to_datetime(activity_data_df['end_date']).dt.date

### Medication Data

In [ ]:
medication_data_df = pd.read_csv(os.path.join(processed_data_path, "20260306_processed_medication_data.csv"))

### Pre-Inclusion Medication Data

In [ ]:
prior_inclusion_medication_df = pd.read_csv(os.path.join(processed_data_path, "20260416_prior_inclusion_medication_data.csv"))

### Examinations Data

In [ ]:
examinations_data_df = pd.read_csv(os.path.join(processed_data_path, "20260306_processed_examinations.csv"))

examinations_data_df['inclusion_date'] = pd.to_datetime(examinations_data_df['inclusion_date']).dt.date
examinations_data_df['endpoint_date'] = pd.to_datetime(examinations_data_df['endpoint_date']).dt.date
examinations_data_df['start_date'] = pd.to_datetime(examinations_data_df['start_date']).dt.date
examinations_data_df['end_date'] = pd.to_datetime(examinations_data_df['end_date']).dt.date

### Social & Behavioural Data

In [ ]:
social_behavioural_data_df = pd.read_csv(os.path.join(processed_data_path, "20260306_social_behavioural_data.csv"))

social_behavioural_data_df['inclusion_date'] = pd.to_datetime(social_behavioural_data_df['inclusion_date']).dt.date
social_behavioural_data_df['endpoint_date'] = pd.to_datetime(social_behavioural_data_df['endpoint_date']).dt.date

### MedCAT Model Data

In [ ]:
medcat_ouput_df = pd.read_csv(os.path.join(data_path, 'processed_data', '20260206_CKD_Comorbidites_Model_Results.csv'))

## Create Full CKD DataFrame

### Create Base Table

In [ ]:
join_cols =['master_person_id', 'inclusion_date', 'endpoint_date']
drop_cols = ['patient_identifier3', 'patient_identifier4', 'patient_identifier2', 'raceDetailed']

base_df = inclusion_patients_df.merge(dates_breakdown_df, how='left', on=join_cols).drop(columns=drop_cols)

del inclusion_patients_df, dates_breakdown_df, join_cols, drop_cols

### Add Social & Behavioural Data

In [ ]:
join_cols =['master_person_id', 'inclusion_date', 'endpoint_date']

add_social_behavioural_df = base_df.merge(social_behavioural_data_df, how='left', on=join_cols)

del base_df, social_behavioural_data_df, join_cols

### Add Comorbidities

In [ ]:
add_comorbidities_df = add_social_behavioural_df.merge(medcat_ouput_df, how='left', on='master_person_id')

comorbidity_col_idx = add_comorbidities_df.columns.get_loc("Hypertensive_disorder_systemic_arterial")
comorbidity_columns = add_comorbidities_df.columns[comorbidity_col_idx:]

for col in comorbidity_columns:
    add_comorbidities_df[col] = add_comorbidities_df[col].apply(lambda x: 0 if pd.isnull(x) else int(x))

del add_social_behavioural_df, medcat_ouput_df, comorbidity_col_idx, comorbidity_columns, col

### Add Activity Data

In [ ]:
join_cols = ['master_person_id', 'inclusion_date', 'endpoint_date', 'grouping', 'start_date', 'end_date']

add_activity_df = add_comorbidities_df.merge(activity_data_df, how='left', on=join_cols)

del add_comorbidities_df, activity_data_df, join_cols

### Add Medication Data

In [ ]:
add_medication_df = add_activity_df.merge(medication_data_df, how='left', on=['master_person_id', 'grouping'])

med_col_idx = add_medication_df.columns.get_loc("ARB")
medication_columns = add_medication_df.columns[med_col_idx:]

for col in medication_columns:
    add_medication_df[col] = add_medication_df[col].apply(lambda x: 0 if pd.isnull(x) else int(x))

del add_activity_df, medication_data_df, med_col_idx, medication_columns, col

### Add Pre-Inclusion Medication Data

In [ ]:
add_preinclusion_meds_df = add_medication_df.merge(prior_inclusion_medication_df, how='left', on='master_person_id')

med_col_idx = add_preinclusion_meds_df.columns.get_loc("priorARB")
medication_columns = add_preinclusion_meds_df.columns[med_col_idx:]

for col in medication_columns:
    add_preinclusion_meds_df[col] = add_preinclusion_meds_df[col].apply(lambda x: 0 if pd.isnull(x) else int(x))

del add_medication_df, prior_inclusion_medication_df, med_col_idx, medication_columns, col

### Add Examinations Data

In [ ]:
join_cols = ['master_person_id', 'inclusion_date', 'endpoint_date', 'grouping', 'start_date', 'end_date']

add_examinations_df = add_preinclusion_meds_df.merge(examinations_data_df, how='left', on=join_cols)

del add_preinclusion_meds_df, examinations_data_df, join_cols

### Add Lab Results Data

In [ ]:
join_cols = ['master_person_id', 'inclusion_date', 'endpoint_date', 'grouping', 'start_date', 'end_date']

add_lab_results_df = add_examinations_df.merge(lab_results_df, how='left', on=join_cols)

del add_examinations_df, lab_results_df, join_cols

## Structure Full DataFrame

### Include only Relevant Endpoint & Dialysis Catchment Patients

In [ ]:
endpoint_filter = (add_lab_results_df['endpoint_type'].isin(['Dialysis', np.NaN]))
districts_filter = (add_lab_results_df['gstt_dialysis_districts']==1)

refined_dataset_df = add_lab_results_df[endpoint_filter&districts_filter].reset_index(drop=True)

del add_lab_results_df, endpoint_filter, districts_filter

### Final Structuring

In [ ]:
cols = ['master_person_id', 'dateOfBirth', 'dateOfDeath', 'gender', 'race', 'imd_decile', 'imd_rank', 'postcode', 'postcode_district', 'postcode_region',
        'gstt_dialysis_districts', 'gfr_slope_districts', 'inclusion_date', 'endpoint_date', 'endpoint_type', 'endpoint_code', 'endpoint_code_type',
        'access_details', 'lives_alone', 'occupation', 'smoking_status', 'heavy_alcohol_use', 'Hypertensive_disorder_systemic_arterial', 'Diabetes_mellitus',
        'Heart_disease', 'Ischemic_heart_disease', 'Heart_failure', 'Cerebrovascular_disease', 'Cerebrovascular_accident', 'Mental_disorder', 'Depressive_disorder',
        'Anxiety', 'Schizophrenia', 'Addiction', 'Frailty', 'Peripheral_vascular_disease', 'Acute_renal_failure_syndrome', 'Chronic_kidney_disease',
        'Peripheral_edema', 'Peripheral_nerve_disease', 'Visual_impairment', 'Impaired_mobility', 'Dyspnea', 'Retinopathy_due_to_diabetes_mellitus',
        'Pulmonary_edema', 'Ulcer_of_foot_due_to_diabetes_mellitus', 'Myocardial_infarction', 'Organic_mental_disorder', 'Bardet-Biedl_syndrome', 'Dialysis_procedure',
        'grouping', 'start_date', 'end_date', 'ae_activities_count', 'ic_activities_count', 'op_attended_count', 'op_dna_count', 'op_cancelled_count',
        'aki_activities_count', 'ARB', 'Ace_Inhibitors', 'Beta_Blocker', 'CCB', 'Clopidogrel', 'DPP4', 'Doxazosin', 'Entresto', 'Ezetimibe', 'GLP1', 'Gliclazide',
        'Hydralazine', 'Insulin', 'Isosorbide', 'MRA', 'Metformin', 'Minoxidil', 'Moxonidine', 'SLGT2i', 'Statin', 'priorARB', 'priorAce_Inhibitors', 'priorBeta_Blocker',
        'priorCCB', 'priorClopidogrel', 'priorDPP4', 'priorDoxazosin', 'priorEntresto', 'priorEzetimibe', 'priorGLP1', 'priorGliclazide', 'priorHydralazine',
        'priorInsulin', 'priorIsosorbide', 'priorMRA', 'priorMetformin', 'priorMinoxidil', 'priorMoxonidine', 'priorSLGT2i', 'priorStatin', 'BMI', 'Arterial_Pressure',
        'BP_Diastolic', 'BP_Systolic', 'Height', 'Weight', 'Albumin', 'CRP', 'Cholesterol', 'Creatinine', 'Haemoglobin', 'HbA1c', 'NT_proBNP', 'Random_Glucose',
        'Troponin_I', 'Urine_ACR', 'Urine_PCR', 'eGFR', 'eGFR_Slope', 'gfr_calculated', 'GFR_Calculated_Slope', 'kfre']

rename_dict = {}

for col in refined_dataset_df.columns:
    if '_' in col:
        new_name = ''
        for item in col.split('_'):
            if len(item) <= 3:
                new_name += item.upper()
            else:
                new_name += item.title()
        rename_dict[col] = new_name

rename_dict['end_date'] = 'EndDate'
rename_dict['ae_activities_count'] = 'AccidentAndEmergencyActivitiesCount'
rename_dict['ic_activities_count'] = 'IntensiveCareActivitiesCount'
rename_dict['op_attended_count'] = 'OutpatientAttendanceCount'
rename_dict['op_dna_count'] = 'OutpatientDNACount'
rename_dict['op_cancelled_count'] = 'OutpatientCancelledCount'
rename_dict['Hypertensive_disorder_systemic_arterial'] = 'HypertensiveDisorder'
rename_dict['Retinopathy_due_to_diabetes_mellitus'] = 'RetinopathyDueToDiabetesMellitus'
rename_dict['Ulcer_of_foot_due_to_diabetes_mellitus'] = 'UlcerOfFootDueToDiabetesMellitus'
rename_dict['Bardet-Biedl_syndrome'] = 'BardetBiedlSyndrome'
rename_dict['NT_proBNP'] = 'NTproBNP'
rename_dict['gstt_dialysis_districts'] = 'GSTTDialysisDistricts'
rename_dict['eGFR_Slope'] = 'eGFRSlope'
rename_dict['kfre'] = 'KFRE'
rename_dict['priorAce_Inhibitors'] = 'priorAceInhibitors'
rename_dict['priorBeta_Blocker'] = 'priorBetaBlocker'

In [ ]:
refined_dataset_df = refined_dataset_df[cols].rename(columns=rename_dict)

del cols, rename_dict

refined_dataset_df.insert(3, 'AgeAtInclusion', refined_dataset_df.apply(lambda row: relativedelta(row['InclusionDate'], row['dateOfBirth']).years, axis=1))
refined_dataset_df.insert(4, 'AgeAtEndpoint', refined_dataset_df.apply(lambda row: relativedelta(row['EndpointDate'], row['dateOfBirth']).years, axis=1))
refined_dataset_df.insert(5, 'AgeAtDeath', refined_dataset_df.apply(lambda row: np.NaN if row['dateOfDeath'] is pd.NaT else relativedelta(row['dateOfDeath'], row['dateOfBirth']).years, axis=1))

refined_dataset_df.head()

In [ ]:
def years_difference(date1, date2):
    relative_delta = relativedelta(date1, date2)
    
    return int(abs(relative_delta.years + relative_delta.months/12 + relative_delta.days/365))

In [ ]:
def measure_difference(measure1, measure2):
    return measure2-measure1

In [ ]:
def change_per_year(measure1, measure2, date1, date2):
    if not pd.isnull(measure1) and not pd.isnull(measure2):
        val_diff = measure_difference(measure1, measure2)
        year_diff = years_difference(date1, date2)
        return val_diff/year_diff

In [ ]:
blood_tests = ['BMI', 'Albumin', 'CRP', 'Cholesterol', 'Creatinine', 'Haemoglobin', 'HbA1c', 'NTproBNP', 'RandomGlucose',
               'TroponinI', 'UrineACR', 'UrinePCR', 'eGFR', 'GFRCalculated', 'ArterialPressure', 'BPDiastolic', 'BPSystolic']

num = len(blood_tests)
j = 1

for test in blood_tests:
    cols = ['MasterPersonID', 'StartDate', test]
    
    test_df = refined_dataset_df[refined_dataset_df[test].notna()][cols]
    
    upper_lim = refined_dataset_df[refined_dataset_df[test].notna()]['MasterPersonID'].value_counts().max()
    
    for i in range(1, upper_lim):
        temp_df = test_df[cols].shift(-i)
        new_cols = [f'{col}_shifted_{i}' for col in cols]
        temp_df.columns = new_cols
    
        test_df = test_df.join(temp_df)

    del i
    
    test_df = test_df.rename(columns={test: f'{test}_shifted_0', 'StartDate': 'StartDate_shifted_0'})
    
    for i in range(1, upper_lim):
        test_df[f'MasterPersonID_shifted_{i}'] = test_df.apply(lambda row: row[f'MasterPersonID_shifted_{i}'] if row[f'MasterPersonID_shifted_{i}']==row['MasterPersonID'] else np.NaN, axis=1)
        test_df[f'StartDate_shifted_{i}'] = test_df.apply(lambda row: row[f'StartDate_shifted_{i}'] if row[f'MasterPersonID_shifted_{i}']==row['MasterPersonID'] else np.NaN, axis=1)
        test_df[f'{test}_shifted_{i}'] = test_df.apply(lambda row: row[f'{test}_shifted_{i}'] if row[f'MasterPersonID_shifted_{i}']==row['MasterPersonID'] else np.NaN, axis=1)

    del i
    
    drop_cols = [col for col in test_df.columns if 'Master' in col and 'shifted' in col]
    
    test_df = test_df.groupby('MasterPersonID').first().reset_index().drop(columns=drop_cols)
    
    for i in range(upper_lim-1):
        test_df[f'diff{i+1}'] = test_df.apply(lambda row: change_per_year(row[f'{test}_shifted_{i}'], row[f'{test}_shifted_{i+1}'], row[f'StartDate_shifted_{i}'], row[f'StartDate_shifted_{i+1}']), axis=1)

    del i
    
    mean_cols = [col for col in test_df.columns if 'diff' in col]
    keep_cols = ['MasterPersonID', f'{test}MeanChange']
    
    test_df[f'{test}MeanChange'] = test_df[mean_cols].mean(axis=1)
    
    test_df = test_df[keep_cols]
    
    refined_dataset_df = refined_dataset_df.merge(test_df, how='left', on='MasterPersonID')

    del test_df, mean_cols, keep_cols, drop_cols, cols

    print(f'Mean Change for {test} Complete ({j}/{num} - {j/num:.2%}) ✅')

    j+=1

In [ ]:
def gfr_classification(gfr):
    classification = np.NAN

    if gfr >= 90:
        classification = 'G1'
    elif 90 > gfr >= 60:
        classification = 'G2'
    elif 60 > gfr >= 45:
        classification = 'G3a'
    elif 45 > gfr >= 30:
        classification = 'G3b'
    elif 30 > gfr >= 15:
        classification = 'G4'
    elif gfr < 15:
        classification = 'G5'

    return classification

In [ ]:
refined_dataset_df['eGFRClassification'] = refined_dataset_df['eGFR'].apply(lambda gfr: gfr_classification(gfr))

In [ ]:
refined_dataset_df['CalculatedGFRClassification'] = refined_dataset_df['GFRCalculated'].apply(lambda gfr: gfr_classification(gfr))

In [ ]:
def blood_pressure_categorisation(row):
    diastolic = row['BPDiastolic']
    systolic = row['BPSystolic']

    bp_category = pd.NA
    
    if diastolic > 135 or systolic > 85:
        bp_category = 'High'
    elif 120 > systolic <= 135 and 80 > diastolic <= 85:
        bp_category = 'Slightly Raised'
    elif 90 > systolic <= 120 and 60 > diastolic <= 80:
        bp_category = 'Healthy'
    elif systolic <= 90 and diastolic <= 60:
        bp_category = 'Low'

    return bp_category

In [ ]:
refined_dataset_df['BPCategorisation'] = refined_dataset_df.apply(lambda row: blood_pressure_categorisation(row), axis=1)

In [ ]:
def bmi_categorisation(row):
    bmi = row['BMI']

    bmi_category = pd.NA
    
    if bmi < 18.5:
        bmi_category = 'Underweight'
    elif bmi < 25:
        bmi_category = 'Healthy'
    elif bmi < 30:
        bmi_category = 'Overweight'
    elif bmi >= 35:
        bmi_category = 'Obese'

    return bmi_category

In [ ]:
refined_dataset_df['BMICategorisation'] = refined_dataset_df.apply(lambda row: bmi_categorisation(row), axis=1)

In [ ]:
cols = ['MasterPersonID', 'dateOfBirth', 'dateOfDeath', 'AgeAtInclusion', 'AgeAtEndpoint', 'AgeAtDeath', 'gender', 'race', 'IMDDecile', 'IMDRank', 'postcode', 'PostcodeDistrict',
        'PostcodeRegion', 'GSTTDialysisDistricts', 'GFRSlopeDistricts', 'InclusionDate', 'EndpointDate', 'EndpointType', 'EndpointCode', 'EndpointCodeType', 'AccessDetails',
        'LivesAlone', 'occupation', 'SmokingStatus', 'HeavyAlcoholUSE', 'HypertensiveDisorder', 'DiabetesMellitus', 'HeartDisease', 'IschemicHeartDisease', 'HeartFailure',
        'CerebrovascularDisease', 'CerebrovascularAccident', 'MentalDisorder', 'DepressiveDisorder', 'Anxiety', 'Schizophrenia', 'Addiction', 'Frailty', 'PeripheralVascularDisease',
        'AcuteRenalFailureSyndrome', 'ChronicKidneyDisease', 'PeripheralEdema', 'PeripheralNerveDisease', 'VisualImpairment', 'ImpairedMobility', 'Dyspnea',
        'RetinopathyDueToDiabetesMellitus', 'PulmonaryEdema', 'UlcerOfFootDueToDiabetesMellitus', 'MyocardialInfarction', 'OrganicMentalDisorder', 'BardetBiedlSyndrome',
        'DialysisProcedure', 'grouping', 'StartDate', 'EndDate', 'AccidentAndEmergencyActivitiesCount', 'IntensiveCareActivitiesCount', 'OutpatientAttendanceCount',
        'OutpatientDNACount', 'OutpatientCancelledCount', 'AKIActivitiesCount', 'ARB', 'ACEInhibitors', 'BetaBlocker', 'CCB', 'Clopidogrel', 'DPP4', 'Doxazosin', 'Entresto',
        'Ezetimibe', 'GLP1', 'Gliclazide', 'Hydralazine', 'Insulin', 'Isosorbide', 'MRA', 'Metformin', 'Minoxidil', 'Moxonidine', 'SLGT2i', 'Statin', 'priorARB', 'priorAceInhibitors',
        'priorBetaBlocker', 'priorCCB', 'priorClopidogrel', 'priorDPP4', 'priorDoxazosin', 'priorEntresto', 'priorEzetimibe', 'priorGLP1', 'priorGliclazide', 'priorHydralazine',
        'priorInsulin', 'priorIsosorbide', 'priorMRA', 'priorMetformin', 'priorMinoxidil', 'priorMoxonidine', 'priorSLGT2i', 'priorStatin', 'BMI', 'BMICategorisation', 'BMIMeanChange',
        'ArterialPressure', 'ArterialPressureMeanChange', 'BPDiastolic', 'BPDiastolicMeanChange', 'BPSystolic', 'BPSystolicMeanChange', 'BPCategorisation', 'Height', 'Weight',
        'Albumin', 'AlbuminMeanChange', 'CRP', 'CRPMeanChange', 'Cholesterol', 'CholesterolMeanChange', 'Creatinine', 'CreatinineMeanChange', 'Haemoglobin', 'HaemoglobinMeanChange',
        'HbA1c', 'HbA1cMeanChange', 'NTproBNP', 'NTproBNPMeanChange',  'RandomGlucose', 'RandomGlucoseMeanChange', 'TroponinI', 'TroponinIMeanChange', 'UrineACR', 'UrineACRMeanChange',
        'UrinePCR', 'UrinePCRMeanChange', 'eGFR', 'eGFRSlope', 'eGFRClassification', 'eGFRMeanChange', 'GFRCalculated', 'GFRCalculatedSlope', 'CalculatedGFRClassification',
        'GFRCalculatedMeanChange', 'KFRE']

refined_dataset_df = refined_dataset_df[cols]

del cols

refined_dataset_df.head()

## Export Refined Combined Dataset

In [ ]:
# --- Save Results ---
file_name = "20260313_ckd_data_for_fe.csv"

refined_dataset_df.to_csv(os.path.join(data_path, file_name), index=False)
print("✅ Results saved.")

## Sandbox